# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AsmaAssa2471/my-flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### 1. Rule & Reason Code

**Rule:**
If a page has high impressions (>100) but low CTR and is stale (>180 days), flag it for content refresh.

* **Reason Code:** STALE_HIGH_IMP_LOW_CTR
* **Action:** CONTENT_REFRESH
* **Score:** min(1.0, (gsc_impressions / 1000) * (1 - gsc_ctr))

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
import duckdb
import pandas as pd
import numpy as np
import os

con = duckdb.connect()

# 1. Main warehouse table setup (Mock data generation for execution)
con.execute("""
CREATE TABLE IF NOT EXISTS main_warehouse_table AS
SELECT
    'hash_' || CAST(range AS VARCHAR) AS content_hash_id,
    '2026-03-01' AS report_date,
    CAST(RANDOM() * 2000 AS INT) AS gsc_impressions,
    RANDOM() * 0.05 AS gsc_ctr,
    CAST(RANDOM() * 300 AS INT) AS days_since_last_update
FROM range(1, 101);
""")

# 2. Execute baseline ranking query
rule_query = """
SELECT
    content_hash_id,
    report_date,
    LEAST(1.0, (gsc_impressions / 1000.0) * (1.0 - gsc_ctr)) AS score,
    'STALE_HIGH_IMP_LOW_CTR' AS reason_code,
    'CONTENT_REFRESH' AS action_label
FROM main_warehouse_table
WHERE gsc_impressions >= 100
ORDER BY score DESC;
"""

ranked_queue = con.execute(rule_query).df()

# 3. Create output directory and save CSV
os.makedirs('../outputs', exist_ok=True)
ranked_queue.to_csv('../outputs/baseline_action_score.csv', index=False)

print("=== QUESTION 2 EXECUTED SUCCESSFULLY ===")
print("Saved to: work/outputs/baseline_action_score.csv")
print("\nFirst 5 Ranked Rows:")
print(ranked_queue.head())

=== QUESTION 2 EXECUTED SUCCESSFULLY ===
Saved to: work/outputs/baseline_action_score.csv

First 5 Ranked Rows:
  content_hash_id report_date  score             reason_code     action_label
0          hash_1  2026-03-01    1.0  STALE_HIGH_IMP_LOW_CTR  CONTENT_REFRESH
1          hash_3  2026-03-01    1.0  STALE_HIGH_IMP_LOW_CTR  CONTENT_REFRESH
2          hash_5  2026-03-01    1.0  STALE_HIGH_IMP_LOW_CTR  CONTENT_REFRESH
3          hash_8  2026-03-01    1.0  STALE_HIGH_IMP_LOW_CTR  CONTENT_REFRESH
4         hash_12  2026-03-01    1.0  STALE_HIGH_IMP_LOW_CTR  CONTENT_REFRESH


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

| Rank | Action | Reason Code | What Would Make It Wrong |
| :--- | :--- | :--- | :--- |
| **1** | CONTENT_REFRESH | STALE_HIGH_IMP_LOW_CTR | Direct answer snippet on Google SERP (Zero-click query). |
| **2** | CONTENT_REFRESH | STALE_HIGH_IMP_LOW_CTR | Temporary technical indexing issue. |
| **3** | CONTENT_REFRESH | STALE_HIGH_IMP_LOW_CTR | Page is an intentional historical archive. |
| **4** | CONTENT_REFRESH | STALE_HIGH_IMP_LOW_CTR | Seasonal query drop in search volume. |
| **5** | CONTENT_REFRESH | STALE_HIGH_IMP_LOW_CTR | Title tag mismatch on search engine page. |
| **6** | CONTENT_REFRESH | STALE_HIGH_IMP_LOW_CTR | Search intent shifted for the primary keyword. |
| **7** | CONTENT_REFRESH | STALE_HIGH_IMP_LOW_CTR | Competitor launched a dominant new feature/tool. |
| **8** | CONTENT_REFRESH | STALE_HIGH_IMP_LOW_CTR | Mobile formatting issue lowering CTR. |
| **9** | CONTENT_REFRESH | STALE_HIGH_IMP_LOW_CTR | Evergreen topic that needs no factual updates. |
| **10** | CONTENT_REFRESH | STALE_HIGH_IMP_LOW_CTR | Internal navigation link changes. |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### 4. Weak Picks & Leakage Check

* **Weak Picks:** Zero-click searches where users get answers directly from Google snippets naturally have low CTR. Flagging them is a weak pick.
* **Leakage Check:** No product flags or future test-set dates were used in feature inputs.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.